# Rec Systems IV
# Item-Based Collaborative Filtering

This workbook is part of a series, listed below:
- <a href="https://nbviewer.org/github/pw598/Articles/blob/main/Rec%20Systems%20I%20-%20Baseline%20Methods.ipynb">Rec Systems I - Baseline Methods</a>
- <a href="https://nbviewer.org/github/pw598/Articles/blob/main/Rec%20Systems%20II%20-%20Content%20Based.ipynb">Rec Systems II - Content Based</a>
- <a href="https://nbviewer.org/github/pw598/Articles/blob/main/Rec%20Systems%20III%20-%20User-Based%20Collaborative%20Filtering.ipynb">Rec Systems III - User-Based Collaborative Filtering</a>
- <a href="https://nbviewer.org/github/pw598/Articles/blob/main/Rec%20Systems%20IV%20-%20Item-Based%20Collaborative%20Filtering.ipynb">Rec Systems IV - Item-Based Collaborative Filtering</a>
- <a href="https://nbviewer.org/github/pw598/Articles/blob/main/Rec%20Systems%20V%20-%20Matrix%20Factorization.ipynb">Rec Systems V - Matrix Factorization</a>

As you might imagine, the code for item-based collaborative can be accomplished with similar code to the user-based filtering, simply by switching the positions of 'user' and 'movie' in the functions and tables.

We'll import the libraries and data.

In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

import warnings 
warnings.filterwarnings('ignore')

In [2]:
movie_info = pd.read_csv('movielens_data/movie_info.csv',header=0,sep=",")
movie_info.head(1)

,movie id,movie title,release date,unknown,Action,Adventure,Animation,Children's,Comedy,Crime,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-95,0,0,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
ratings = pd.read_csv('movielens_data/ratings.csv',header=0,sep=",")
ratings.drop(columns='unix_timestamp', inplace=True)
ratings.head(1)

,user_id,movie_id,rating
0,196,242,3


In [4]:
user_info = pd.read_csv('movielens_data/user_demographics.csv',header=0,sep=",")
user_info.head(1)

,user_id,age,sex,occupation,zip_code
0,1,24,M,technician,85711


# Error Metrics

In [5]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# Train/Test Split

In [21]:
X = ratings.copy()
X_train, X_test = train_test_split(X, test_size = 0.25, random_state=123)

# Item-Based Collaborative Filtering

We'll start by determining the ratings matrix.

In [7]:
r_matrix = X_train.pivot_table(values='rating', index='movie_id', columns='user_id')
r_matrix.head()

user_id,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
movie_id,,,,,,,,,,,,,,,,,,,,,
1,NaN,4.0,NaN,NaN,4.0,4.0,NaN,NaN,NaN,4.0,...,2.0,3.0,4.0,NaN,4.0,NaN,NaN,5.0,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Then the cosine similarity amongst items.

In [8]:
r_matrix_dummy = r_matrix.copy().fillna(0)
cosine_sim = cosine_similarity(r_matrix_dummy, r_matrix_dummy)
cosine_sim = pd.DataFrame(cosine_sim, index=r_matrix.index, columns=r_matrix.index)
cosine_sim.head(5)

movie_id,1,2,3,4,5,6,7,8,9,10,...,1672,1673,1674,1675,1676,1677,1678,1679,1680,1681
movie_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.306973,0.231494,0.314676,0.169864,0.088947,0.434730,0.349151,0.361484,0.200839,...,0.038626,0.0,0.0,0.000000,0.000000,0.040969,0.0,0.0,0.0,0.054626
2,0.306973,1.000000,0.218471,0.375924,0.247366,0.072077,0.247149,0.280675,0.166096,0.119424,...,0.064700,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.091499
3,0.231494,0.218471,1.000000,0.231121,0.133669,0.056202,0.275855,0.138467,0.231465,0.106376,...,0.000000,0.0,0.0,0.000000,0.000000,0.037503,0.0,0.0,0.0,0.000000
4,0.314676,0.375924,0.231121,1.000000,0.283467,0.086192,0.362702,0.354373,0.348474,0.210962,...,0.048063,0.0,0.0,0.113286,0.113286,0.000000,0.0,0.0,0.0,0.067971
5,0.169864,0.247366,0.133669,0.283467,1.000000,0.016770,0.235328,0.197037,0.207065,0.040368,...,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000


Then the predict function.

In [9]:
def predict(user, movie):
    dummy = r_matrix[user].fillna(0)
    sim_scores = cosine_sim[movie]
    pred = sum(sim_scores * dummy) / sim_scores[dummy > 0].sum()
    return pred

### Prediction Upon Training Set

In [12]:
p_matrix = r_matrix.copy()

for user in tqdm(r_matrix.columns):
    for movie in r_matrix.index:
        if not pd.isnull(r_matrix[user][movie]):
            pred = predict(user, movie)
            p_matrix[user][movie] = pred

100%|██████████| 943/943 [01:44<00:00,  8.98it/s]


In [13]:
preds = []
actuals = []

for user in tqdm(p_matrix.columns):
    for movie in p_matrix.index:
        if not pd.isnull(p_matrix[user][movie]):
            preds.append(p_matrix[user][movie])
            actuals.append(r_matrix[user][movie])

100%|██████████| 943/943 [00:15<00:00, 62.76it/s]


The training RMSE is as follows:

In [14]:
rmse(actuals, preds)

0.9504515045456705

### Prediction Upon Test Set

In [15]:
r_matrix_test = X_test.pivot_table(values='rating', index='movie_id', columns='user_id')
r_matrix_test.head(2)

user_id,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
movie_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3.0,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
print(r_matrix.shape)
print(r_matrix_test.shape)

(1641, 943)
(1454, 943)


In [18]:
p_matrix = r_matrix_test.copy()

for user in tqdm(r_matrix_test.columns):
    for movie in r_matrix_test.index:
        if not pd.isnull(r_matrix_test[user][movie]):
            try:
                pred = predict(user, movie)
                p_matrix[user][movie] = pred
            except:
                pass

100%|██████████| 943/943 [00:45<00:00, 20.59it/s]


In [19]:
preds = []
actuals = []

for user in tqdm(p_matrix.columns):
    for movie in p_matrix.index:
        if not pd.isnull(p_matrix[user][movie]):
            preds.append(p_matrix[user][movie])
            actuals.append(r_matrix_test[user][movie])

100%|██████████| 943/943 [00:13<00:00, 70.34it/s]


The test RMSE is as follows:

In [20]:
rmse(actuals, preds)

1.0230187355775586

In [24]:
results_df = pd.DataFrame({"model":['Simple Average', 'Naive User-Based Avg', 
                                    'Naive Item-Based Avg', 'Vote-Weighted Avg', 'IMDB Weighted Avg', 
                                    'User-Based CF'],
                        "train_rmse":[1.123, 1.025, 0.995, 1.123, 1.036, 0.931], 
                        "test_rmse":[1.134, 1.054, 1.028, 1.134, 1.052, 1.020]})

results_df

,model,train_rmse,test_rmse
0,Simple Average,1.123,1.134
1,Naive User-Based Avg,1.025,1.054
2,Naive Item-Based Avg,0.995,1.028
3,Vote-Weighted Avg,1.123,1.134
4,IMDB Weighted Avg,1.036,1.052
5,User-Based CF,0.931,1.020


In [25]:
def append_results_df(model, train_rmse, test_rmse):
    new_row = len(results_df) + 1
    results_df.loc[new_row,'model'] = model
    results_df.loc[new_row,'train_rmse'] = round(train_rmse,3)
    results_df.loc[new_row,'test_rmse'] = round(test_rmse,3)

In [27]:
append_results_df('Item-Based CF', 0.950, 1.023)
results_df

,model,train_rmse,test_rmse
0,Simple Average,1.123,1.134
1,Naive User-Based Avg,1.025,1.054
2,Naive Item-Based Avg,0.995,1.028
3,Vote-Weighted Avg,1.123,1.134
4,IMDB Weighted Avg,1.036,1.052
5,User-Based CF,0.931,1.020
7,Item-Based CF,0.950,1.023


The next workbook in the series looks at matrix factorization, and can be found <a href="https://nbviewer.org/github/pw598/Articles/blob/main/Rec%20Systems%20V%20-%20Matrix%20Factorization.ipynb">here</a>.